# EDA — Image editing by visual demonstration (S → Sp, T → T')
Notebook di analisi esplorativa orientato al task: **localizzazione** (30%), **trasferimento semantico dell'edit** (50%), **preservazione dello sfondo** (20%).

Struttura: 0 Config · 1 Integrità · 2 Maschere · 3 Natura dell'edit S→Sp · 4 Similarità S↔T · 5 CLIP (Δ_S, tipologie) · 6 Localizzazione col modello fine-tuned · 7 Baseline di scoring · 8 Riepilogo (JSON)

Ogni sezione stampa righe `[INSIGHT:...]`; a fine notebook c'è un riepilogo unico e un file `eda_summary.json`.
Le sezioni con modelli (CLIP, DINOv3 fine-tuned, LPIPS) si disattivano con i flag in Config e non bloccano il resto.

## 0. Config (path fittizi: da sostituire)

In [ ]:
import os, json, math, glob, warnings, copy, time, random
from pathlib import Path
from collections import OrderedDict, Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as TF
warnings.filterwarnings("ignore")

# ============================ CONFIG ============================
DATA_DIR     = os.environ.get("AML_DATA", "/kaggle/input/NOME-COMPETIZIONE")                       # contiene train/ e test/ (anche annidati)
DINO_WEIGHTS = os.environ.get("AML_DINO_L", "/kaggle/input/models/NOME-UTENTE/dinov3-l16/pytorch/default/1/dinov3-vit-l16.pth")
HEAD_CKPT    = os.environ.get("AML_HEAD", "/kaggle/input/models/NOME-UTENTE/headL/pytorch/default/1/headL_full_ep6.pt")
CLIP_DIR     = os.environ.get("AML_CLIP", "openai/clip-vit-large-patch14")  # "" -> scarica "openai/clip-vit-base-patch32"
OUT_DIR      = os.environ.get("AML_OUT", "/kaggle/working")
USE_LOCALIZER, USE_CLIP, USE_LPIPS = True, True, True
N_PTS        = 8          # punti campionati dentro MS per il localizzatore (+ il centroide)
ABLATE_ZOOM  = False      # True: valuta anche il fine-tuned senza zoom (più lento)
CHANGE_THR   = 25         # soglia (0-255) per considerare "cambiato" un pixel S vs Sp
SEED = 0
# ================================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(f"{OUT_DIR}/figs", exist_ok=True)
INSIGHTS, RESULTS = [], {}
def say(sec, txt): print(f"[INSIGHT:{sec}] {txt}"); INSIGHTS.append(dict(section=sec, text=txt))
def savefig(name): plt.tight_layout(); plt.savefig(f"{OUT_DIR}/figs/{name}.png", dpi=110, bbox_inches="tight"); plt.show()
plt.rcParams.update({"figure.dpi": 90, "axes.grid": True, "grid.alpha": .25})
print("device:", DEVICE)

## 1. Integrità e struttura

In [ ]:
def find_split(name):
    c = [p for p in Path(DATA_DIR).rglob(name) if p.is_dir() and any(p.glob("example_*"))]
    assert c, f"split '{name}' non trovato sotto {DATA_DIR}"
    return sorted(c, key=lambda p: len(p.parts))[0]
SPLIT_DIR = {s: find_split(s) for s in ("train", "test")}; print(SPLIT_DIR)
EXPECTED = {"train": ["S.png", "Sp.png", "T.png", "MS.png", "MT.png"], "test": ["S.png", "Sp.png", "T.png", "MS.png"]}
RGB_F = ("S.png", "Sp.png", "T.png")
rows, DATA = [], {}
for split, d in SPLIT_DIR.items():
    for ex in tqdm(sorted(d.glob("example_*")), desc=split):
        names = {f.name for f in ex.iterdir()}; rec = dict(key=f"{split}/{ex.name}", split=split, id=ex.name)
        rec["missing"] = ",".join(f for f in EXPECTED[split] if f not in names); item = {"MT": None}
        for f in EXPECTED[split]:
            if f not in names: continue
            im = Image.open(ex / f); rec[f[:-4] + "_mode"] = im.mode; rec[f[:-4] + "_size"] = str(im.size)
            if f in RGB_F: item[f[:-4]] = im.convert("RGB")
            else:
                a = np.array(im.convert("L")); rec[f[:-4] + "_nonbinary"] = bool(set(np.unique(a)) - {0, 255}); item[f[:-4]] = a > 127
        DATA[rec["key"]] = item; rows.append(rec)
INT = pd.DataFrame(rows).set_index("key")
for split in ("train", "test"):
    s = INT[INT.split == split]
    print(f"{split}: {len(s)} esempi | mancanti: {(s.missing != '').sum()} | modi S/T/MS: {s.S_mode.unique().tolist()}/{s.T_mode.unique().tolist()}/{s.MS_mode.unique().tolist()}")
    print("   dimensioni S:", s.S_size.value_counts().to_dict(), "| T:", s["T_size"].value_counts().to_dict())
    print("   maschere non binarie (MS):", int(s.MS_nonbinary.sum()), "| (MT):", int(s.MT_nonbinary.sum()) if "MT_nonbinary" in s else "n/a")
ids_tr, ids_te = set(INT[INT.split == "train"].id), set(INT[INT.split == "test"].id)
print("ID in comune train/test:", len(ids_tr & ids_te), "| range id:", min(ids_tr | ids_te), "→", max(ids_tr | ids_te))
say("integrita", f"train={len(ids_tr)}, test={len(ids_te)}, mancanti={int((INT.missing != '').sum())}, maschere non binarie={int(INT.filter(like='nonbinary').sum().sum())}, id sovrapposti={len(ids_tr & ids_te)}")

TRAIN_KEYS = [k for k in DATA if k.startswith("train/")]; TEST_KEYS = [k for k in DATA if k.startswith("test/")]
def overlay(img, m, color=(255, 0, 0), a=.5):
    x = np.asarray(img).astype(np.float32).copy()
    if m is not None and m.any(): x[m] = (1 - a) * x[m] + a * np.array(color, np.float32)
    return x.astype(np.uint8)
def gallery(keys, title="", maxn=6):
    keys = list(keys)[:maxn]; fig, ax = plt.subplots(len(keys), 5, figsize=(13, 2.7 * len(keys))); ax = np.atleast_2d(ax)
    for r, k in enumerate(keys):
        d = DATA[k]; panels = [("S", d["S"]), ("Sp", d["Sp"]), ("T", d["T"]), ("S + MS", overlay(d["S"], d["MS"])),
                               ("T + MT" if d["MT"] is not None else "T (no MT)", overlay(d["T"], d["MT"], (0, 255, 0)) if d["MT"] is not None else d["T"])]
        for c, (t, im) in enumerate(panels):
            ax[r, c].imshow(im); ax[r, c].axis("off"); ax[r, c].set_title(f"{t}" + (f"  {k}" if c == 0 else ""), fontsize=8)
    plt.suptitle(title, y=1.0); savefig("gallery_" + title.replace(" ", "_")[:30])
gallery(random.Random(SEED).sample(TRAIN_KEYS, min(6, len(TRAIN_KEYS))), "esempi casuali train")

## 2. Maschere: area, forma, posizione, relazione MS ↔ MT

In [ ]:
from scipy import ndimage as ndi
from scipy.stats import ks_2samp, spearmanr
def mstats(m, p):
    area = int(m.sum())
    if area == 0: return {f"{p}_{k}": np.nan for k in ("frac", "cx", "cy", "bw", "bh", "ncc", "main_cc", "extent", "border")} | {f"{p}_area": 0}
    ys, xs = np.nonzero(m); lab, n = ndi.label(m); sz = np.bincount(lab.ravel())[1:]; bw, bh = xs.max() - xs.min() + 1, ys.max() - ys.min() + 1
    return {f"{p}_area": area, f"{p}_frac": area / m.size, f"{p}_cx": xs.mean(), f"{p}_cy": ys.mean(), f"{p}_bw": bw, f"{p}_bh": bh, f"{p}_ncc": n,
            f"{p}_main_cc": sz.max() / area, f"{p}_extent": area / (bw * bh), f"{p}_border": float(xs.min() == 0 or ys.min() == 0 or xs.max() == m.shape[1] - 1 or ys.max() == m.shape[0] - 1)}
rows = []
for k, d in DATA.items():
    r = dict(key=k, split=k.split("/")[0]) | mstats(d["MS"], "MS")
    if d["MT"] is not None: r |= mstats(d["MT"], "MT")
    rows.append(r)
DF = pd.DataFrame(rows).set_index("key"); TR = DF[DF.split == "train"]; TE = DF[DF.split == "test"]
TR_ = TR.assign(area_ratio=TR.MT_area / TR.MS_area.clip(lower=1), dcx=TR.MT_cx - TR.MS_cx, dcy=TR.MT_cy - TR.MS_cy)
TR_["shift"] = np.hypot(TR_.dcx, TR_.dcy); DF = DF.join(TR_[["area_ratio", "dcx", "dcy", "shift"]])
print(DF.groupby("split")[["MS_frac", "MS_ncc", "MS_main_cc", "MS_extent", "MS_border"]].describe().T.round(3).to_string())
print("\nMT (train):"); print(TR[["MT_frac", "MT_ncc", "MT_main_cc", "MT_extent", "MT_border"]].describe().round(3).T.to_string())

fig, ax = plt.subplots(2, 3, figsize=(15, 8))
b = np.logspace(-3.5, 0, 30); ax[0, 0].hist(TR.MS_frac, b, alpha=.6, label="MS train"); ax[0, 0].hist(TE.MS_frac, b, alpha=.6, label="MS test"); ax[0, 0].hist(TR.MT_frac, b, alpha=.4, label="MT train")
ax[0, 0].set_xscale("log"); ax[0, 0].set_title("frazione di pixel della maschera"); ax[0, 0].legend()
ax[0, 1].scatter(TR.MS_frac, TR.MT_frac, s=14); ax[0, 1].plot([1e-3, 1], [1e-3, 1], "k--"); ax[0, 1].set_xscale("log"); ax[0, 1].set_yscale("log"); ax[0, 1].set_xlabel("MS frac"); ax[0, 1].set_ylabel("MT frac"); ax[0, 1].set_title("area MS vs MT")
ax[0, 2].hist(np.log2(TR_.area_ratio.replace(0, np.nan).dropna()), 25); ax[0, 2].set_title("log2(area MT / area MS)"); ax[0, 2].axvline(0, c="k")
for a_, (x, y, t) in zip(ax[1, :2], [(TR.MS_cx, TR.MS_cy, "centroidi MS"), (TR.MT_cx, TR.MT_cy, "centroidi MT")]):
    h = a_.hist2d(x.dropna(), y.dropna(), bins=8, range=[[0, 256], [0, 256]], cmap="viridis"); a_.invert_yaxis(); a_.set_title(t + " (train)"); a_.grid(False)
ax[1, 2].scatter(TR_.dcx, TR_.dcy, s=14); ax[1, 2].axhline(0, c="k"); ax[1, 2].axvline(0, c="k"); ax[1, 2].invert_yaxis(); ax[1, 2].set_title("spostamento centroide MS→MT (px)")
savefig("masks_overview")

c = pd.Series(np.hypot(TR.MS_cx - 127.5, TR.MS_cy - 127.5)); 
L_same = np.exp(-TR_["shift"] / (0.1 * np.sqrt(2 * 255 ** 2))); L_ctr = np.exp(-np.hypot(TR.MT_cx - 127.5, TR.MT_cy - 127.5) / (0.1 * np.sqrt(2 * 255 ** 2)))
ks = ks_2samp(TR.MS_frac, TE.MS_frac)
say("maschere", f"MS area mediana train={TR.MS_frac.median():.3%}, test={TE.MS_frac.median():.3%} (KS p={ks.pvalue:.2f}); MS con >1 componente: {(TR.MS_ncc > 1).mean():.0%}; MS che toccano il bordo: {TR.MS_border.mean():.0%}")
say("maschere", f"rapporto area MT/MS: mediana {TR_.area_ratio.median():.2f} (IQR {TR_.area_ratio.quantile(.25):.2f}–{TR_.area_ratio.quantile(.75):.2f}); spostamento centroide mediano {TR_['shift'].median():.0f}px")
say("maschere", f"baseline spaziali sul train → L(stessa posizione di MS)={L_same.mean():.3f}, L(centro immagine)={L_ctr.mean():.3f}")
RESULTS["L_same_pos"], RESULTS["L_center"] = float(L_same.mean()), float(L_ctr.mean())

## 3. Natura dell'edit S → Sp

In [ ]:
from skimage.metrics import structural_similarity as ssim
from skimage import color as skc
def edit_stats(d):
    S, Sp, M = np.asarray(d["S"]).astype(np.float32), np.asarray(d["Sp"]).astype(np.float32), d["MS"]
    diff = np.abs(S - Sp).max(-1); ch = diff > CHANGE_THR; nch = int(ch.sum()); reg = (ch & M) if (ch & M).any() else (ch if nch else M)
    o = dict(ch_frac=nch / ch.size, diff_in=diff[M].mean() if M.any() else np.nan, diff_out=diff[~M].mean(), in_ratio=(ch & M).sum() / max(nch, 1), cover=(ch & M).sum() / max(M.sum(), 1),
             ssim_ss=ssim(S.mean(-1), Sp.mean(-1), data_range=255))
    Ls, Lp = skc.rgb2lab(S / 255), skc.rgb2lab(Sp / 255)
    if reg.any():
        o["dE"] = np.linalg.norm(Lp[reg] - Ls[reg], axis=1).mean(); o["dL"] = Lp[reg, 0].mean() - Ls[reg, 0].mean()
        o["dchroma"] = np.hypot(*(Lp[reg, 1:].mean(0) - Ls[reg, 1:].mean(0)))
        g1, g2 = Ls[reg, 0], Lp[reg, 0]; o["lum_corr"] = np.corrcoef(g1, g2)[0, 1] if g1.std() > 1e-3 and g2.std() > 1e-3 else np.nan
        hs, hp = skc.rgb2hsv(S / 255)[reg, 0] * 2 * np.pi, skc.rgb2hsv(Sp / 255)[reg, 0] * 2 * np.pi
        o["dhue_deg"] = abs(np.degrees(np.angle(np.exp(1j * (np.angle(np.exp(1j * hp).mean()) - np.angle(np.exp(1j * hs).mean()))))))
    return o
def edit_type(r):
    if r.ch_frac < 0.002: return "none"
    if r.ch_frac > 0.5: return "global"
    if r.in_ratio < 0.5: return "outside_mask"
    if r.dchroma > 10 and (r.lum_corr if not np.isnan(r.lum_corr) else 0) > 0.7: return "recolor"
    return "structural"
ES = pd.DataFrame({k: edit_stats(d) for k, d in tqdm(DATA.items(), desc="edit stats")}).T.astype(float)
ES["type"] = ES.apply(edit_type, axis=1); DF = DF.join(ES)
print(DF.groupby("split")[["ch_frac", "in_ratio", "cover", "diff_in", "diff_out", "ssim_ss", "dE", "dchroma", "lum_corr"]].median().round(3).T.to_string())
print("\ntipo di edit (euristico):"); print(pd.crosstab(DF.type, DF.split).to_string())

fig, ax = plt.subplots(2, 3, figsize=(15, 8))
ax[0, 0].hist(DF.ch_frac.clip(1e-5), np.logspace(-5, 0, 30)); ax[0, 0].set_xscale("log"); ax[0, 0].set_title("frazione di pixel cambiati (S vs Sp)")
ax[0, 1].hist(DF.in_ratio.dropna(), 20); ax[0, 1].set_title("quota dei pixel cambiati che cade dentro MS")
ax[0, 2].hist(DF.cover.dropna(), 20); ax[0, 2].set_title("quota di MS effettivamente modificata")
for t, g in DF.groupby("type"): ax[1, 0].scatter(g.dchroma, g.lum_corr, s=16, label=f"{t} ({len(g)})")
ax[1, 0].set_xlabel("Δ chroma Lab"); ax[1, 0].set_ylabel("corr. luminanza S/Sp nella regione"); ax[1, 0].legend(fontsize=8); ax[1, 0].set_title("recolor vs strutturale")
ax[1, 1].scatter(DF.MS_frac, DF.ch_frac.clip(1e-5), s=14); ax[1, 1].set_xscale("log"); ax[1, 1].set_yscale("log"); ax[1, 1].set_xlabel("MS frac"); ax[1, 1].set_ylabel("ch frac"); ax[1, 1].plot([1e-3, 1], [1e-3, 1], "k--"); ax[1, 1].set_title("area MS vs area cambiata")
ax[1, 2].hist(DF.dE.dropna(), 25); ax[1, 2].set_title("intensità media dell'edit (ΔE Lab)")
savefig("edit_nature")

def gallery_diff(keys, title, maxn=4):
    keys = list(keys)[:maxn]
    if not keys: return
    fig, ax = plt.subplots(len(keys), 4, figsize=(11, 2.7 * len(keys))); ax = np.atleast_2d(ax)
    for r, k in enumerate(keys):
        d = DATA[k]; df_ = np.abs(np.asarray(d["S"]).astype(float) - np.asarray(d["Sp"]).astype(float)).max(-1)
        for c, (t, im, cm) in enumerate([("S", d["S"], None), ("Sp", d["Sp"], None), ("|S-Sp|", df_, "magma"), ("S+MS", overlay(d["S"], d["MS"]), None)]):
            ax[r, c].imshow(im, cmap=cm); ax[r, c].axis("off"); ax[r, c].set_title(t + (f" {k}" if c == 0 else ""), fontsize=8)
    plt.suptitle(title, y=1.0); savefig("diff_" + title.replace(" ", "_")[:30])
for t in ("none", "outside_mask", "global", "recolor", "structural"):
    ks_ = DF[(DF.type == t)].index.tolist()
    if ks_: gallery_diff(random.Random(SEED).sample(ks_, min(4, len(ks_))), f"tipo {t}")
n_all = len(DF); sh = DF.type.value_counts(normalize=True)
say("edit", "tipi di edit (euristici): " + ", ".join(f"{t}={v:.0%}" for t, v in sh.items()))
say("edit", f"edit anomali (none/outside_mask/global): train={int(DF[DF.split=='train'].type.isin(['none','outside_mask','global']).sum())}, test={int(DF[DF.split=='test'].type.isin(['none','outside_mask','global']).sum())}")
say("edit", f"ΔE mediano nella regione cambiata={DF.dE.median():.1f}; quota cambiata dentro MS mediana={DF.in_ratio.median():.0%}; copertura di MS mediana={DF.cover.median():.0%}")
say("edit", "anomali test: " + ", ".join(DF[(DF.split == 'test') & DF.type.isin(['none', 'outside_mask', 'global'])].index.str.replace('test/', '').tolist()[:30]))

## 4. Similarità S ↔ T (quanto è dura la corrispondenza?)

In [ ]:
def hs_hist(img):
    h = skc.rgb2hsv(np.asarray(img) / 255.); H, _, _ = np.histogram2d(h[..., 0].ravel(), h[..., 1].ravel(), bins=(16, 8), range=[[0, 1], [0, 1]]); return (H / H.sum()).ravel()
def part_lab(img, m): return skc.rgb2lab(np.asarray(img) / 255.)[m].mean(0) if m.any() else np.full(3, np.nan)
rows = {}
for k, d in tqdm(DATA.items(), desc="S↔T"):
    hs, ht = hs_hist(d["S"]), hs_hist(d["T"]); gS, gT = np.asarray(d["S"]).astype(float).mean(-1), np.asarray(d["T"]).astype(float).mean(-1)
    r = dict(hist_corr=np.corrcoef(hs, ht)[0, 1], ssim_st=ssim(gS, gT, data_range=255), pix_mad=np.abs(gS - gT).mean())
    if d["MT"] is not None: r["part_dE_st"] = np.linalg.norm(part_lab(d["S"], d["MS"]) - part_lab(d["T"], d["MT"]))
    rows[k] = r
ST = pd.DataFrame(rows).T.astype(float); DF = DF.join(ST)
print(DF.groupby("split")[["hist_corr", "ssim_st", "pix_mad", "part_dE_st"]].describe().T.round(3).to_string())
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(DF.hist_corr.dropna(), 25); ax[0].set_title("correlazione istogramma H-S (S vs T)"); ax[1].hist(DF.ssim_st, 25); ax[1].set_title("SSIM S vs T (alto = stessa scena/layout)")
ax[2].scatter(DF.ssim_st, DF.hist_corr, s=14); ax[2].set_xlabel("SSIM"); ax[2].set_ylabel("hist corr"); savefig("s_vs_t")
same_scene = (DF.ssim_st > 0.5).mean()
say("S_vs_T", f"SSIM S/T mediano={DF.ssim_st.median():.2f}; coppie con layout molto simile (SSIM>0.5): {same_scene:.0%}; correlazione colore mediana={DF.hist_corr.median():.2f}")
say("S_vs_T", "se SSIM è alto la coppia è quasi la stessa scena (matching facile, copia diretta della geometria); se basso è un'istanza diversa della categoria (serve matching semantico)")
gallery(DF[DF.split == "train"].sort_values("ssim_st").index[:3].tolist() + DF[DF.split == "train"].sort_values("ssim_st").index[-3:].tolist(), "S-T meno e piu simili")

## 5. CLIP: intensità Δ_S, tipologie di edit, casi anomali

In [ ]:
CLIP = None
if USE_CLIP:
    try:
        from transformers import CLIPModel, CLIPProcessor
        src = CLIP_DIR or "openai/clip-vit-base-patch32"
        CLIP = CLIPModel.from_pretrained(src).to(DEVICE).eval(); CLIPP = CLIPProcessor.from_pretrained(src); print("CLIP caricato da", src)
    except Exception as e: print("CLIP non disponibile → sezione saltata:", repr(e)[:200]); CLIP = None
def _t(f): return f if torch.is_tensor(f) else f.pooler_output
@torch.no_grad()
def clip_img(imgs, bs=64):
    out = []
    for i in range(0, len(imgs), bs):
        x = CLIPP(images=imgs[i:i + bs], return_tensors="pt")["pixel_values"].to(DEVICE); out.append(TF.normalize(_t(CLIP.get_image_features(pixel_values=x)).float(), dim=-1).cpu())
    return torch.cat(out).numpy()
@torch.no_grad()
def clip_txt(txts):
    x = CLIPP(text=txts, return_tensors="pt", padding=True).to(DEVICE); return TF.normalize(_t(CLIP.get_text_features(**x)).float(), dim=-1).cpu().numpy()
def bbox_crop(img, m, margin=.1, minsz=32):
    if not m.any(): return img
    ys, xs = np.nonzero(m); x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1; pw, ph = max(minsz - (x1 - x0), 0) / 2 + margin * (x1 - x0), max(minsz - (y1 - y0), 0) / 2 + margin * (y1 - y0)
    return img.crop((int(max(0, x0 - pw)), int(max(0, y0 - ph)), int(min(img.width, x1 + pw)), int(min(img.height, y1 + ph))))   # NB: ritaglio "attorno alla maschera" = approssimazione della metrica ufficiale
DS, CLIPF = {}, None
if CLIP is not None:
    keys = list(DATA); eS = clip_img([bbox_crop(DATA[k]["S"], DATA[k]["MS"]) for k in keys]); eSp = clip_img([bbox_crop(DATA[k]["Sp"], DATA[k]["MS"]) for k in keys])
    gS = clip_img([DATA[k]["S"] for k in keys]); gT = clip_img([DATA[k]["T"] for k in keys])
    tk = [k for k in keys if DATA[k]["MT"] is not None]; eT_part = dict(zip(tk, clip_img([bbox_crop(DATA[k]["T"], DATA[k]["MT"]) for k in tk])))
    dS = eSp - eS; dn = np.linalg.norm(dS, axis=1); DS = {k: dS[i] for i, k in enumerate(keys)}
    CLIPF = pd.DataFrame(dict(dS_norm=dn, cos_part_S_Sp=(eS * eSp).sum(1), cos_glob_ST=(gS * gT).sum(1)), index=keys)
    CLIPF["cos_part_ST"] = pd.Series({k: float(eS[keys.index(k)] @ eT_part[k]) for k in tk})
    DF = DF.join(CLIPF)
    print(DF.groupby("split")[["dS_norm", "cos_part_S_Sp", "cos_glob_ST", "cos_part_ST"]].describe().T.round(3).to_string())
    # --- tipologia dell'edit: direzione di Δ_S vs direzioni testuali ---
    cols = ["red", "blue", "green", "yellow", "orange", "pink", "purple", "black", "white", "brown", "gray"]; tex = ["metallic", "wooden", "furry", "glossy", "rusty", "striped", "spotted", "transparent", "shiny", "dirty"]
    oth = ["a bigger object", "a smaller object", "a broken object", "a smiling face", "an open object", "a closed object", "a bent object", "an object with a pattern"]
    P = {**{f"colore:{c}": f"a {c} object" for c in cols}, **{f"texture:{t}": f"a {t} object" for t in tex}, **{f"altro:{o}": o for o in oth}}
    te_ = clip_txt(list(P.values())); dirs = te_ - te_.mean(0); dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)
    cs = (dS / np.maximum(dn[:, None], 1e-8)) @ dirs.T; top = np.array(list(P))[cs.argmax(1)]; DF["clip_dir"] = pd.Series(top, index=keys); DF["clip_dir_family"] = DF.clip_dir.str.split(":").str[0]
    print("\nfamiglia direzione CLIP di Δ_S:", DF.clip_dir_family.value_counts(normalize=True).round(2).to_dict()); print(DF.clip_dir.value_counts().head(12).to_string())
    # --- PCA + KMeans sui Δ_S normalizzati ---
    from sklearn.decomposition import PCA; from sklearn.cluster import KMeans; from sklearn.metrics import silhouette_score
    X = dS / np.maximum(dn[:, None], 1e-8); Z = PCA(2, random_state=0).fit_transform(X); Xp = PCA(20, random_state=0).fit_transform(X)
    sil = {k_: silhouette_score(Xp, KMeans(k_, n_init=5, random_state=0).fit_predict(Xp)) for k_ in range(2, 9)}; K_ = max(sil, key=sil.get); km = KMeans(K_, n_init=10, random_state=0).fit(Xp); DF["clip_cluster"] = pd.Series(km.labels_, index=keys)
    fig, ax = plt.subplots(1, 4, figsize=(18, 4)); ax[0].hist(dn, 30); ax[0].set_title("|Δ_S| in spazio CLIP (intensità edit)")
    sc_ = ax[1].scatter(Z[:, 0], Z[:, 1], c=km.labels_, cmap="tab10", s=16); ax[1].set_title(f"PCA di Δ_S/|Δ_S| (KMeans k={K_}, sil={sil[K_]:.2f})")
    ax[2].scatter(DF.dE, DF.dS_norm, s=14, c=(DF.index.str.startswith("test")).astype(int), cmap="coolwarm"); ax[2].set_xlabel("ΔE pixel"); ax[2].set_ylabel("|Δ_S| CLIP"); ax[2].set_title("intensità pixel vs CLIP (rosso=test)")
    DF.groupby("type").dS_norm.median().plot.bar(ax=ax[3]); ax[3].set_title("|Δ_S| mediano per tipo euristico"); savefig("clip_edit")
    for c_ in range(K_):
        mem = DF[DF.clip_cluster == c_].index; idx = np.argsort(np.linalg.norm(Xp[[keys.index(m) for m in mem]] - km.cluster_centers_[c_], axis=1))[:3]; gallery_diff([mem[i] for i in idx], f"cluster CLIP {c_} (n={len(mem)})", 3)
    rho = spearmanr(DF.dS_norm, DF.dE, nan_policy="omit")[0]; low = DF.dS_norm < DF.dS_norm.quantile(.05)
    say("clip", f"|Δ_S| mediano={DF.dS_norm.median():.3f} (IQR {DF.dS_norm.quantile(.25):.3f}–{DF.dS_norm.quantile(.75):.3f}); Spearman con ΔE pixel={rho:.2f}")
    say("clip", f"famiglie di edit (direzione CLIP): " + ", ".join(f"{a}={b:.0%}" for a, b in DF.clip_dir_family.value_counts(normalize=True).items()) + f"; cluster ottimali k={K_} (silhouette {sil[K_]:.2f})")
    say("clip", f"5% con |Δ_S| più basso (edit probabilmente non riusciti): {int(low.sum())} esempi; di questi flaggati dalle euristiche pixel (none/outside/global): {int((low & DF.type.isin(['none','outside_mask','global'])).sum())}")
    say("clip", f"similarità CLIP parte S ↔ parte T (train) mediana={DF.cos_part_ST.median():.2f}; globale S↔T mediana={DF.cos_glob_ST.median():.2f}")
    RESULTS["dS_norm_median"] = float(DF.dS_norm.median())

## 6. Localizzazione col tuo modello (DINOv3-L fine-tuned) vs baseline

In [ ]:
# ---- definizioni copiate dalla pipeline di training (stessa architettura/config); input = immagini PIL in memoria ----
IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32); IMNET_STD = np.array([0.229, 0.224, 0.225], np.float32)
DINO_CFG = dict(hidden_size=1024, intermediate_size=4096, num_hidden_layers=24, num_attention_heads=16, patch_size=16, num_register_tokens=4)
K_TRAIN, UP, CANON = 4, 2, 128
F2_CFG = dict(sizes=(512, 768), decode="soft", tau=.03, rad=3, zoom=dict(margin=.3, max_frac=.9, sizes=(512, 768)))
_HF = {}
def load_dino():
    if "m" in _HF: return _HF["m"]
    from transformers import DINOv3ViTConfig, DINOv3ViTModel
    m = DINOv3ViTModel(DINOv3ViTConfig(**DINO_CFG))
    if DINO_WEIGHTS.endswith(".safetensors"):
        from safetensors.torch import load_file; sd = load_file(DINO_WEIGHTS)
    else:
        sd = torch.load(DINO_WEIGHTS, map_location="cpu"); sd = sd["state_dict"] if isinstance(sd, dict) and "state_dict" in sd else sd
    own = set(m.state_dict().keys())
    variants = [sd, {("model." + k if k.startswith("layer.") else k): v for k, v in sd.items()}, {(k[6:] if k.startswith("model.") else k): v for k, v in sd.items()}]
    best = min(variants, key=lambda d: len(own - set(d.keys()))); miss, unexp = m.load_state_dict(best, strict=False); print("chiavi mancanti:", miss[:6], "| inattese:", unexp[:6])
    assert not [k for k in miss if "layer" in k or "embeddings" in k], "pesi DINOv3 non caricati!"
    m = m.to(DEVICE).float().eval()
    for p in m.parameters(): p.requires_grad_(False)
    _HF["m"] = m; return m
def locate_parts(m):
    name = next((n for n, mod in m.named_modules() if isinstance(mod, nn.ModuleList) and len(mod) == m.config.num_hidden_layers), None); assert name is not None
    parent = m.get_submodule(name.rsplit(".", 1)[0]) if "." in name else m; layers = m.get_submodule(name)
    norm = getattr(parent, "norm", None) or next(mod for n, mod in m.named_modules() if n.endswith("norm") and isinstance(mod, nn.LayerNorm))
    rope = getattr(parent, "rope_embeddings", None) or next(mod for n, mod in m.named_modules() if "rope" in n.lower()); return layers, norm, rope
class Trunk:
    P = 16
    def __init__(s, K=K_TRAIN):
        s.m = load_dino(); s.layers, s.norm, s.rope_mod = locate_parts(s.m); s.L = len(s.layers); s.K = K; s.R = int(s.m.config.num_register_tokens); s.mem = OrderedDict(); s._pe = {}
    def grid(s, W, H, size): sc = size / max(W, H); return max(1, round(H * sc / s.P)), max(1, round(W * sc / s.P))
    def rope(s, gh, gw):
        if (gh, gw) not in s._pe:
            with torch.no_grad(): pe = s.rope_mod(torch.zeros(1, 3, gh * s.P, gw * s.P, device=DEVICE))
            s._pe[(gh, gw)] = tuple(t.float() for t in pe)
        return s._pe[(gh, gw)]
    @torch.no_grad()
    def tokens(s, img, size, key):
        k = (key, size)
        if k in s.mem: s.mem.move_to_end(k); return s.mem[k]
        W, H = img.size; gh, gw = s.grid(W, H, size); x = (np.asarray(img.resize((gw * s.P, gh * s.P), Image.BICUBIC), np.float32) / 255. - IMNET_MEAN) / IMNET_STD
        x = torch.from_numpy(x.transpose(2, 0, 1))[None].to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")): o = s.m(pixel_values=x, output_hidden_states=True)
        out = (o.hidden_states[s.L - s.K][0].float(), gh, gw); s.mem[k] = out
        if len(s.mem) > 48: s.mem.popitem(last=False)
        return out
class Head(nn.Module):
    def __init__(s, trunk, up=UP):
        super().__init__(); C = trunk.m.config.hidden_size; s.layers = copy.deepcopy(trunk.layers[trunk.L - trunk.K:]).float(); s.norm = copy.deepcopy(trunk.norm).float(); s.R, s.up = trunk.R, up
        s.refine = nn.Sequential(nn.Conv2d(C, 128, 1), nn.GELU(), nn.Conv2d(128, 128, 3, padding=1, groups=128), nn.GELU(), nn.Conv2d(128, C, 1))
    def forward(s, tok, pe, gh, gw):
        h = tok
        for l in s.layers:
            o = l(h, position_embeddings=pe); h = o[0] if isinstance(o, (tuple, list)) else o
        h = s.norm(h)[:, 1 + s.R:]; B, N, C = h.shape; assert N == gh * gw
        f = TF.normalize(h.transpose(1, 2).reshape(B, C, gh, gw), dim=1)
        if s.up > 1: f = TF.interpolate(f, scale_factor=s.up, mode="bilinear", align_corners=False); f = TF.normalize(f + s.refine(f), dim=1)
        return f
def sample_desc(F, pts, W, H):
    C, h, w = F.shape; p = np.asarray(pts, np.float32)
    u = torch.as_tensor(p[:, 0] / W * w - .5, device=F.device); v = torch.as_tensor(p[:, 1] / H * h - .5, device=F.device)
    g = torch.stack([2 * u / max(w - 1, 1) - 1, 2 * v / max(h - 1, 1) - 1], -1)[None, None]
    return TF.normalize(TF.grid_sample(F[None], g, align_corners=True, padding_mode="border")[0, :, 0].T, dim=-1)
class Engine:
    def __init__(s, head, trunk, max_cache=12): s.head, s.trunk, s.lru, s.max = head.eval(), trunk, OrderedDict(), max_cache
    @torch.no_grad()
    def F(s, img, size, key):
        k = (key, size)
        if k in s.lru: s.lru.move_to_end(k); return s.lru[k]
        t, gh, gw = s.trunk.tokens(img, size, key); f = s.head(t[None], s.trunk.rope(gh, gw), gh, gw)[0]; s.lru[k] = f
        if len(s.lru) > s.max: s.lru.popitem(last=False)
        return f
def canon_shape(Wt, Ht): sc = CANON / max(Wt, Ht); return max(2, round(Ht * sc)), max(2, round(Wt * sc))
@torch.no_grad()
def view_map(eng, pr, size):
    Ws, Hs = pr["S"].size; Wt, Ht = pr["T"].size; Fs, Ft = eng.F(pr["S"], size, pr["kS"]), eng.F(pr["T"], size, pr["kT"]); C, h, w = Ft.shape
    S = (sample_desc(Fs, pr["pts"], Ws, Hs) @ Ft.reshape(C, -1)).reshape(-1, 1, h, w); Hc, Wc = canon_shape(Wt, Ht)
    return TF.interpolate(S, size=(Hc, Wc), mode="bicubic", align_corners=False)[:, 0]
def decode_soft(S, Wt, Ht, tau=.03, rad=3):
    Hc, Wc = S.shape; i0, j0 = np.unravel_index(np.argmax(S), S.shape); i1, i2, j1, j2 = max(0, i0 - rad), min(Hc, i0 + rad + 1), max(0, j0 - rad), min(Wc, j0 + rad + 1)
    w = np.exp((S[i1:i2, j1:j2] - S[i0, j0]) / tau); w /= w.sum(); ii, jj = np.meshgrid(np.arange(i1, i2), np.arange(j1, j2), indexing="ij")
    return (((w * jj).sum() + .5) / Wc * Wt, ((w * ii).sum() + .5) / Hc * Ht)
@torch.no_grad()
def predict_pair(pr, eng, cfg):
    Wt, Ht = pr["T"].size; maps = (sum(view_map(eng, pr, sz) for sz in cfg["sizes"]) / len(cfg["sizes"])).cpu().numpy()
    return np.array([decode_soft(S, Wt, Ht, cfg.get("tau", .03), cfg.get("rad", 3)) for S in maps])
def box_from(pts, W, H, margin):
    pts = np.asarray(pts, float); x0, y0 = pts.min(0); x1, y1 = pts.max(0); pad = margin * max(x1 - x0, y1 - y0, .1 * max(W, H))
    return (int(max(0, x0 - pad)), int(max(0, y0 - pad)), int(min(W, math.ceil(x1 + pad))), int(min(H, math.ceil(y1 + pad))))
def predict_zoom(pr, eng, cfg):
    out = predict_pair(pr, eng, cfg); z = cfg.get("zoom")
    if not z or len(out) < 3: return out
    Ws, Hs = pr["S"].size; Wt, Ht = pr["T"].size; pts = np.asarray(pr["pts"], float)
    core = np.percentile(out, [10, 90], axis=0) if len(out) >= 5 else np.array([out.min(0), out.max(0)]); sb, tb = box_from(pts, Ws, Hs, z["margin"]), box_from(core, Wt, Ht, z["margin"])
    fs, ft = (sb[2] - sb[0]) * (sb[3] - sb[1]) / (Ws * Hs), (tb[2] - tb[0]) * (tb[3] - tb[1]) / (Wt * Ht)
    if max(fs, ft) > z["max_frac"] or min(sb[2] - sb[0], sb[3] - sb[1], tb[2] - tb[0], tb[3] - tb[1]) < 32: return out
    pr2 = dict(S=pr["S"].crop(sb), T=pr["T"].crop(tb), kS=f"{pr['kS']}|{sb}", kT=f"{pr['kT']}|{tb}", pts=(pts - np.array(sb[:2])).astype(np.float32))
    return predict_pair(pr2, eng, dict(cfg, sizes=z["sizes"], zoom=None)) + np.array(tb[:2])

# ---- da maschera MS a punti, e da punti predetti a centroide + maschera stimata ----
def sample_points(ms, n=N_PTS):
    ys, xs = np.nonzero(ms); Pp = np.stack([xs, ys], 1).astype(np.float32); c = Pp.mean(0, keepdims=True)
    if len(Pp) <= n: return np.vstack([c, Pp])
    from sklearn.cluster import KMeans
    C = KMeans(n, n_init=3, random_state=0).fit(Pp).cluster_centers_; C = Pp[((C[:, None] - Pp[None]) ** 2).sum(-1).argmin(1)]; return np.vstack([c, C])
def est_mask(pred, src, ms_area, shape=(256, 256)):
    from skimage.morphology import convex_hull_image, disk
    canvas = np.zeros(shape, bool); P_ = np.clip(np.round(pred).astype(int), 0, [shape[1] - 1, shape[0] - 1]); canvas[P_[:, 1], P_[:, 0]] = True
    def hull_area(Q):
        c = np.zeros(shape, bool); Q = np.clip(np.round(Q).astype(int), 0, [shape[1] - 1, shape[0] - 1]); c[Q[:, 1], Q[:, 0]] = True
        try: return float(convex_hull_image(c).sum())
        except Exception: return 0.
    hp, hs = hull_area(pred), hull_area(src); s = math.sqrt(hp / hs) if hs > 0 and hp > 0 else 1.0; r = max(1, int(.5 * s * math.sqrt(ms_area / len(pred))))
    try: hull = convex_hull_image(canvas)
    except Exception: hull = canvas
    if hull.sum() < 4: return ndi.binary_dilation(canvas, disk(max(2, int(s * math.sqrt(ms_area / math.pi)))))
    return ndi.binary_dilation(hull, disk(r))
def localize(key, eng, cfg=F2_CFG):
    d = DATA[key]; pts = sample_points(d["MS"]); pr = dict(S=d["S"], T=d["T"], kS=key + "/S", kT=key + "/T", pts=pts); out = predict_zoom(pr, eng, cfg)
    out = np.clip(out, 0, 255); cen = np.median(out, 0); return dict(cx=cen[0], cy=cen[1], mask=est_mask(out, pts, int(d["MS"].sum())), pred=out)
LD = 0.1 * math.sqrt(2 * 255 ** 2)
def Lscore(p, t): return float(math.exp(-math.hypot(p[0] - t[0], p[1] - t[1]) / LD))
def iou(a, b): u = (a | b).sum(); return float((a & b).sum() / u) if u else np.nan

In [ ]:
HAVE_LOC, PRED = False, {}          # PRED[model][key] = dict(cx,cy,mask,pred)
if USE_LOCALIZER:
    try:
        TRUNK = Trunk(); HEAD = Head(TRUNK).to(DEVICE); sd = torch.load(HEAD_CKPT, map_location=DEVICE); sd = sd.get("state_dict", sd) if isinstance(sd, dict) else sd
        print(HEAD.load_state_dict(sd)); HEAD.eval(); HEAD0 = Head(TRUNK).to(DEVICE).eval()   # HEAD0 = DINOv3-L originale (testa non addestrata = pesi originali + refine a residuo zero)
        with torch.no_grad(): nn.init.zeros_(HEAD0.refine[-1].weight); nn.init.zeros_(HEAD0.refine[-1].bias)
        HAVE_LOC = True
    except Exception as e: print("Localizzatore non disponibile → sezione saltata:", repr(e)[:300])
def run_loc(name, head, keys, cfg=F2_CFG):
    eng = Engine(head, TRUNK); PRED[name] = {}
    for k in tqdm(keys, desc=name):
        try: PRED[name][k] = localize(k, eng, cfg)
        except Exception as e: print("errore", k, repr(e)[:120])
if HAVE_LOC:
    t0 = time.time(); run_loc("finetuned", HEAD, TRAIN_KEYS); print(f"finetuned/train: {time.time() - t0:.0f}s")
    run_loc("dino_ep0", HEAD0, TRAIN_KEYS)
    if ABLATE_ZOOM: run_loc("finetuned_nozoom", HEAD, TRAIN_KEYS, dict(F2_CFG, zoom=None))
    run_loc("finetuned", HEAD, TEST_KEYS) if False else None
    eng_te = Engine(HEAD, TRUNK); PRED.setdefault("finetuned_test", {})
    for k in tqdm(TEST_KEYS, desc="test"): PRED["finetuned_test"][k] = localize(k, eng_te)

In [ ]:
if HAVE_LOC:
    rows = []
    for k in TRAIN_KEYS:
        d = DATA[k]; tc = (DF.loc[k, "MT_cx"], DF.loc[k, "MT_cy"]); r = dict(key=k)
        base = {"centro": dict(cx=127.5, cy=127.5, mask=None), "stessa_pos": dict(cx=DF.loc[k, "MS_cx"], cy=DF.loc[k, "MS_cy"], mask=None)}
        for n in ("finetuned", "dino_ep0") + (("finetuned_nozoom",) if ABLATE_ZOOM else ()): 
            if k in PRED.get(n, {}): base[n] = PRED[n][k]
        for n, p in base.items():
            r[f"dist_{n}"] = math.hypot(p["cx"] - tc[0], p["cy"] - tc[1]); r[f"L_{n}"] = Lscore((p["cx"], p["cy"]), tc)
            if p["mask"] is not None: r[f"iou_{n}"] = iou(p["mask"], d["MT"])
        rows.append(r)
    LOC = pd.DataFrame(rows).set_index("key"); DF = DF.join(LOC)
    names = [c[2:] for c in LOC.columns if c.startswith("L_")]
    tab = pd.DataFrame({n: dict(L_medio=LOC[f"L_{n}"].mean(), dist_med_px=LOC[f"dist_{n}"].median(), dist_media_px=LOC[f"dist_{n}"].mean(), **{f"≤{t}px": (LOC[f"dist_{n}"] <= t).mean() for t in (10, 25, 50)},
                        IoU_medio=LOC[f"iou_{n}"].mean() if f"iou_{n}" in LOC else np.nan) for n in names}).T.round(3); print(tab.to_string()); RESULTS["localization"] = tab.to_dict()
    fig, ax = plt.subplots(1, 4, figsize=(19, 4))
    for n in names: ax[0].hist(LOC[f"dist_{n}"], np.linspace(0, 200, 40), alpha=.45, label=n)
    ax[0].set_title("errore di localizzazione (px)"); ax[0].legend(fontsize=8)
    ax[1].scatter(DF.loc[TRAIN_KEYS, "MS_frac"], LOC.dist_finetuned, s=14); ax[1].set_xscale("log"); ax[1].set_xlabel("MS frac"); ax[1].set_ylabel("errore px"); ax[1].set_title("errore vs area parte")
    ax[2].scatter(DF.loc[TRAIN_KEYS, "ssim_st"], LOC.dist_finetuned, s=14); ax[2].set_xlabel("SSIM S/T"); ax[2].set_title("errore vs similarità S/T")
    ax[3].scatter(LOC.iou_finetuned, LOC.L_finetuned, s=14); ax[3].set_xlabel("IoU maschera stimata"); ax[3].set_ylabel("L"); ax[3].set_title("IoU vs L"); savefig("localization")
    for lab_, col in [("area MS", "MS_frac"), ("n componenti MS", "MS_ncc"), ("SSIM S/T", "ssim_st"), ("area ratio MT/MS", "area_ratio")]:
        v = DF.loc[TRAIN_KEYS, col]; print(f"Spearman(errore fine-tuned, {lab_}) = {spearmanr(LOC.dist_finetuned, v, nan_policy='omit')[0]:+.2f}")
    worst = LOC.dist_finetuned.sort_values(ascending=False).index[:6]
    fig, ax = plt.subplots(len(worst), 3, figsize=(10, 3.3 * len(worst)))
    for r_, k in enumerate(worst):
        d = DATA[k]; p = PRED["finetuned"][k]; ax[r_, 0].imshow(overlay(d["S"], d["MS"])); ax[r_, 0].set_title(f"{k}  S+MS", fontsize=8)
        im = overlay(overlay(d["T"], d["MT"], (0, 255, 0), .4), p["mask"], (0, 0, 255), .35); ax[r_, 1].imshow(im); ax[r_, 1].plot(p["cx"], p["cy"], "b*", ms=14); ax[r_, 1].plot(DF.loc[k, "MT_cx"], DF.loc[k, "MT_cy"], "g*", ms=14)
        ax[r_, 1].plot(p["pred"][:, 0], p["pred"][:, 1], "y.", ms=6); ax[r_, 1].set_title(f"T: verde=MT, blu=stima, err={LOC.loc[k,'dist_finetuned']:.0f}px", fontsize=8); ax[r_, 2].imshow(d["T"]); [a.axis("off") for a in ax[r_]]
    savefig("localization_failures")
    f_, o_ = LOC.L_finetuned.mean(), LOC.L_dino_ep0.mean() if "L_dino_ep0" in LOC else np.nan
    say("localizzazione", f"L medio (train): fine-tuned={f_:.3f} vs DINOv3 originale={o_:.3f} vs stessa-posizione={LOC.L_stessa_pos.mean():.3f} vs centro={LOC.L_centro.mean():.3f}")
    say("localizzazione", f"fine-tuned: errore mediano={LOC.dist_finetuned.median():.1f}px, ≤25px: {(LOC.dist_finetuned <= 25).mean():.0%}, IoU medio maschera stimata={LOC.iou_finetuned.mean():.2f} (domain shift: il modello è stato addestrato su keypoint, non su maschere)")
    RESULTS["L_finetuned"] = float(f_)
else: say("localizzazione", "sezione saltata (modello non disponibile)")

## 7. Baseline di scoring locale (L, E, B) — riferimento prima di costruire l'editor

In [ ]:
LP = None
if USE_LPIPS:
    try: import lpips; LP = lpips.LPIPS(net="alex", spatial=True, verbose=False).to(DEVICE).eval(); print("LPIPS-alex OK")
    except Exception as e: print("LPIPS non disponibile → B = proxy (|diff| fuori maschera):", repr(e)[:150])
def to_lp(a): return (torch.from_numpy(np.asarray(a).astype(np.float32) / 127.5 - 1).permute(2, 0, 1)[None]).to(DEVICE)
@torch.no_grad()
def B_score(T, Tp, true_mt):
    prot = ~ndi.binary_dilation(true_mt, iterations=3)           # tolleranza 3px come da regolamento
    if LP is not None: D = float(LP(to_lp(T), to_lp(Tp))[0, 0].cpu().numpy()[prot].mean())
    else: D = float(np.abs(np.asarray(T).astype(float) - np.asarray(Tp).astype(float)).mean(-1)[prot].mean() / 255 * 2)
    return math.exp(-D / 0.1)
def E_score(key, Tp_img, true_mt):
    if CLIP is None: return np.nan
    d = DATA[key]; eT, eTp = clip_img([bbox_crop(d["T"], true_mt), bbox_crop(Tp_img, true_mt)]); dT = eTp - eT; dS = DS[key]
    cos = float(dT @ dS / (np.linalg.norm(dT) * np.linalg.norm(dS) + 1e-8)); return max(0.0, cos) * min(1.0, np.linalg.norm(dT) / (np.linalg.norm(dS) + 1e-8))   # riproduzione locale (da verificare contro il testo ufficiale)
def feather(m, px=1.5): return np.clip(ndi.gaussian_filter(m.astype(np.float32), px), 0, 1)[..., None]
def region(d):
    S, Sp = np.asarray(d["S"]).astype(np.float32), np.asarray(d["Sp"]).astype(np.float32); ch = np.abs(S - Sp).max(-1) > CHANGE_THR; r = ch & d["MS"]; return r if r.sum() > 20 else d["MS"]
def apply_lab(d, mask, mode):
    S, Sp, T = [skc.rgb2lab(np.asarray(d[x]) / 255.) for x in ("S", "Sp", "T")]; rs = region(d)
    if not mask.any() or not rs.any(): return d["T"]
    T2 = T.copy()
    if mode == "shift": T2[mask] = T[mask] + (Sp[rs].mean(0) - S[rs].mean(0))
    else: T2[mask] = (T[mask] - T[mask].mean(0)) / (T[mask].std(0) + 1e-3) * Sp[rs].std(0) + Sp[rs].mean(0)      # Reinhard: media/std della regione demo
    T2[..., 0] = np.clip(T2[..., 0], 0, 100); rgb = (np.clip(skc.lab2rgb(T2), 0, 1) * 255); w = feather(mask)
    return Image.fromarray((np.asarray(d["T"]) * (1 - w) + rgb * w).astype(np.uint8))
def total(L, E, B): return .3 * L + .5 * E + .2 * B
def run_baselines(keys, name, getmask):
    rows = []
    for k in tqdm(keys, desc=name):
        d = DATA[k]; mt = d["MT"]; Lk = DF.loc[k, "L_finetuned"] if "L_finetuned" in DF else DF.loc[k, "L_stessa_pos"] if "L_stessa_pos" in DF else np.nan
        mask = getmask(k)
        if name == "identita": Tp = d["T"]
        else: Tp = apply_lab(d, mask, "shift" if "shift" in name else "reinhard")
        E = 0.0 if name == "identita" else E_score(k, Tp, mt); B = B_score(d["T"], Tp, mt); rows.append(dict(key=k, baseline=name, L=Lk, E=E, B=B, total=total(Lk, E, B)))
    return pd.DataFrame(rows)
if "L_finetuned" not in DF and "L_stessa_pos" not in DF:
    DF["L_stessa_pos"] = np.exp(-np.hypot(DF.MS_cx - DF.MT_cx, DF.MS_cy - DF.MT_cy) / LD)
sub = TRAIN_KEYS[:] if len(TRAIN_KEYS) <= 120 else random.Random(SEED).sample(TRAIN_KEYS, 120)
BL = [run_baselines(sub, "identita", lambda k: None)]
for mode in ("shift", "reinhard"):
    BL.append(run_baselines(sub, f"{mode}+MT_oracolo", lambda k: DATA[k]["MT"]))
    if HAVE_LOC: BL.append(run_baselines(sub, f"{mode}+MT_stimata", lambda k: PRED["finetuned"][k]["mask"]))
BL = pd.concat(BL); BL = BL.merge(DF[["type"]], left_on="key", right_index=True)
tab = BL.groupby("baseline")[["L", "E", "B", "total"]].mean().round(3); print(tab.to_string()); RESULTS["baselines"] = tab.to_dict()
best = tab.total.idxmax(); print(f"\nmigliore: {best} → per tipo di edit:"); print(BL[BL.baseline == best].groupby("type")[["E", "B", "total"]].agg(["mean", "count"]).round(3).to_string())
fig, ax = plt.subplots(1, 2, figsize=(13, 4)); tab[["L", "E", "B", "total"]].plot.bar(ax=ax[0]); ax[0].set_title("punteggi medi per baseline (train, locale)"); ax[0].tick_params(axis="x", rotation=30)
bt = BL[BL.baseline == best].groupby("type").total.mean(); bt.plot.bar(ax=ax[1], color="C2"); ax[1].set_title(f"total per tipo di edit ({best})"); savefig("baselines")
say("baseline", "totale locale medio: " + ", ".join(f"{i}={v:.3f}" for i, v in tab.total.items()) + (" [E=NaN: CLIP assente]" if CLIP is None else "") + (" [B proxy: LPIPS assente]" if LP is None else ""))
say("baseline", f"il color-transfer semplice ({best}) ottiene E medio={tab.loc[best,'E']:.3f}: " + (", ".join(f"{t}={v:.2f}" for t, v in BL[BL.baseline == best].groupby("type").E.mean().items()) if CLIP is not None else "n/d"))

## 8. Riepilogo, raccomandazioni, salvataggio

In [ ]:
sh = DF.type.value_counts(normalize=True); rec = []
if sh.get("recolor", 0) >= .4: rec.append(f"{sh.get('recolor',0):.0%} degli edit sono ricolorazioni: una trasformazione diretta in Lab dentro la maschera (stile Reinhard) è una baseline forte e quasi gratuita, con sfondo intatto.")
if sh.get("structural", 0) >= .25: rec.append(f"{sh.get('structural',0):.0%} degli edit sono strutturali/di texture: servono diffusion/flow-matching in inpainting guidato (istruzione da VLM o inversione S→Sp) con guida CLIP sulla direzione Δ_S.")
anom = DF.type.isin(["none", "outside_mask", "global"]).mean()
if anom > .03: rec.append(f"{anom:.0%} degli esempi ha edit anomali (nessuna modifica / fuori maschera / globale): regola dedicata (modifica prudente o T quasi invariata) e verifica con |Δ_S| CLIP.")
if HAVE_LOC:
    if RESULTS.get("L_finetuned", 0) > RESULTS["L_same_pos"] + .1: rec.append("Il localizzatore fine-tuned batte nettamente la baseline 'stessa posizione': usarlo per centroide e per il posizionamento della maschera.")
    else: rec.append("Il localizzatore non batte nettamente la baseline spaziale: valutare prior di posizione o matching patch→patch alternativo.")
    rec.append("La maschera stimata (inviluppo dei punti predetti) è approssimativa: raffinarla (SAM con il punto trovato, o soglia sulla mappa di similarità) e ricomporre sempre T originale fuori maschera con piccolo margine.")
rec.append("Calibrare l'intensità dell'edit su |Δ_S| (CLIP): la metrica non premia modifiche più forti della demo e penalizza quelle più deboli.")
for r_ in rec: say("raccomandazione", r_)
print("\n" + "=" * 70 + "\nRIEPILOGO (copia/incolla questo blocco)\n" + "=" * 70)
for s in INSIGHTS: print(f"- [{s['section']}] {s['text']}")
DF.to_csv(f"{OUT_DIR}/eda_features.csv")
if HAVE_LOC:
    pd.DataFrame([dict(id=k.split("/")[1], x=v["cx"], y=v["cy"]) for k, v in PRED["finetuned_test"].items()]).to_csv(f"{OUT_DIR}/loc_test_finetuned.csv", index=False)
    np.savez_compressed(f"{OUT_DIR}/pred_masks_test.npz", **{k.split("/")[1]: v["mask"] for k, v in PRED["finetuned_test"].items()})
json.dump(dict(insights=INSIGHTS, results=RESULTS), open(f"{OUT_DIR}/eda_summary.json", "w"), indent=1, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o))
print(f"\nsalvati in {OUT_DIR}: eda_summary.json, eda_features.csv" + (", loc_test_finetuned.csv, pred_masks_test.npz" if HAVE_LOC else "") + ", figs/*.png")